In [1]:
import os
import ssl
import certifi
import urllib3

# Configure SSL certificates
cert_path = certifi.where()
os.environ['SSL_CERT_FILE'] = cert_path
os.environ['REQUESTS_CA_BUNDLE'] = cert_path
os.environ['AWS_CA_BUNDLE'] = cert_path
os.environ['CURL_CA_BUNDLE'] = cert_path

# Create SSL context with proper certificates
ssl_context = ssl.create_default_context(cafile=cert_path)
ssl._create_default_https_context = lambda: ssl_context

# Disable SSL warnings (optional)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print(f"✅ SSL certificates configured using: {cert_path}")
print("✅ Environment variables set for AWS, requests, and curl")
print("✅ Ready to make secure HTTPS connections!")

✅ SSL certificates configured using: /Users/manojskr/Documents/Code/GitHub/graphrag-toolkit/.venv/lib/python3.10/site-packages/certifi/cacert.pem
✅ Environment variables set for AWS, requests, and curl
✅ Ready to make secure HTTPS connections!


In [2]:
%reload_ext dotenv
%dotenv ../.env


import os

from graphrag_toolkit.lexical_graph import set_logging_config
from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory

set_logging_config('INFO')

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
    graph_store, 
    vector_store,
    streaming=True
)

response = query_engine.query("What are SLMs mean?")

print(f"""{response.print_response_stream()}

retrieve_ms: {int(response.metadata['retrieve_ms'])}
answer_ms  : {int(response.metadata['answer_ms'])}
total_ms   : {int(response.metadata['total_ms'])}
""")

SLMs refer to Statistical Language Models. The search results indicate that statistical language models, also known as n-gram models, are the earliest form of language models. They view text as a sequence of words and estimate the probability of text as the product of the probabilities of individual words, which are calculated based on word and n-gram counts from text corpora. [Source: data/pdfs/sample_Pdf.pdf (sample_Pdf.pdf, 1, 2, 1)]None

retrieve_ms: 6255
answer_ms  : 1099
total_ms   : 7355



In [3]:
graph_store

NeptuneAnalyticsClient(log_formatting=RedactedGraphQueryLogFormatting(), tenant_id=TenantId(value=None), graph_id='g-ssztyll4x2', config='{}')

In [3]:
response

StreamingResponse(response_gen=<generator object stream_chat_response_to_tokens.<locals>.gen at 0x310c5bc30>, source_nodes=[NodeWithScore(node=TextNode(id_='741b18a5-8dfd-4542-9039-46b6a8cabfbe', embedding=None, metadata={'source': {'sourceId': 'aws::413dc223:2ee4', 'metadata': {'source': 'data/pdfs/QEM-CCR-2410-00001-VR(FQE).pdf', 'page_number': '6', 'total_pages': '11', 'file_name': 'QEM-CCR-2410-00001-VR(FQE).pdf', 'page_label': '6'}}, 'topics': [{'topic': 'Calibration', 'statements': [{'statementId': '99251c88b03df9a0a8dd5ea9ede5c7fb', 'statement': 'Tools may not provide a means recorded in the image for calibrating measurements.', 'facts': ['measurements CALIBRATE USING means'], 'details': 'tools PROVIDE means\nmeans RECORDED IN image', 'chunkId': 'aws::413dc223:2ee4:f14ed5fe', 'score': 0.26, 'statement_str': 'Tools may not provide a means recorded in the image for calibrating measurements. (details: measurements CALIBRATE USING means, tools PROVIDE means, means RECORDED IN image)

In [3]:
for n in response.source_nodes:
    print(n.text)

{
  "source": "data/pdfs/sample_Pdf.pdf (sample_Pdf.pdf, 1, 2, 1)",
  "topic": "LLM Advances and Trends",
  "statements": [
    "The field of LLMs is moving fast, with new findings, models and techniques being published in a matter of months or weeks.",
    "LLMs are large-scale, pre-trained, statistical language models based on neural networks.",
    "This paper gives a timely survey of the recent advances on LLMs.",
    "The four waves of language models are statistical language models, neural language models, pre-trained language models, and LLMs.",
    "The survey aims to prove a valuable and accessible resource for students, researchers and developers.",
    "The recent success of LLMs is an accumulation of decades of research and development of language models.",
    "Language models can be categorized into four waves that have different starting points and velocity.",
    "AI researchers and practitioners often find it challenging to figure out the best recipes to build LLM-powe

In [4]:
response.source_nodes

[NodeWithScore(node=TextNode(id_='aed10201-d18c-47d3-9bba-4eae99d59514', embedding=None, metadata={'source': {'sourceId': 'aws::b8b3e890:fa83', 'metadata': {'source': 'data/pdfs/sample_Pdf.pdf', 'page_number': '1', 'total_pages': '2', 'file_name': 'sample_Pdf.pdf', 'page_label': '1'}}, 'topics': [{'topic': 'LLM Advances and Trends', 'statements': [{'statementId': 'dca1f20d37504307e9fd8168b2d7c620', 'statement': 'The field of LLMs is moving fast, with new findings, models and techniques being published in a matter of months or weeks.', 'facts': ['LLMs PUBLISHED IN weeks', 'LLMs DESCRIBED BY new findings', 'LLMs DESCRIBED BY new models', 'LLMs DESCRIBED BY new techniques', 'LLMs PUBLISHED IN months'], 'details': 'LLMs MOVING FAST', 'chunkId': 'aws::b8b3e890:fa83:5626e655', 'score': 0.12, 'statement_str': 'The field of LLMs is moving fast, with new findings, models and techniques being published in a matter of months or weeks. (details: LLMs PUBLISHED IN weeks, LLMs DESCRIBED BY new findi

In [5]:
import json

print("--- Detailed Response Breakdown ---")

# The response.source_nodes contains the list of retrieved information bundles.
for i, node_with_score in enumerate(response.source_nodes):
    node = node_with_score.node
    overall_score = node_with_score.score
    
    print(f"\n[Retrieved Block {i+1}]")
    print(f"  Overall Block Score: {overall_score if overall_score is not None else 'N/A'}")
    
    # The metadata contains the rich, structured information.
    metadata = node.metadata
    source_info = metadata.get('source', {}).get('metadata', {})
    
    print(f"  Source: {source_info.get('file_name', 'Unknown')}, Page: {source_info.get('page_number', 'N/A')}")
    print("-" * 30)

    # Each block can contain multiple topics.
    for topic_info in metadata.get('topics', []):
        print(f"  Topic: {topic_info.get('topic')}")
        
        # Sort the statements within this topic by their individual score.
        sorted_statements = sorted(
            topic_info.get('statements', []), 
            key=lambda x: x.get('score', 0), 
            reverse=True
        )
        
        if not sorted_statements:
            print("    - No statements found for this topic.")
            continue

        # Print each statement with its score.
        for statement in sorted_statements:
            statement_text = statement.get('statement')
            statement_score = statement.get('score', 0)
            print(f"    - [Relevance: {statement_score:.4f}] {statement_text}")

print("\n--- End of Breakdown ---")

--- Detailed Response Breakdown ---

[Retrieved Block 1]
  Overall Block Score: N/A
  Source: sample_Pdf.pdf, Page: 1
------------------------------
  Topic: LLM Advances and Trends
    - [Relevance: 0.1200] The field of LLMs is moving fast, with new findings, models and techniques being published in a matter of months or weeks.
    - [Relevance: 0.0800] LLMs are large-scale, pre-trained, statistical language models based on neural networks.
    - [Relevance: 0.0600] This paper gives a timely survey of the recent advances on LLMs.
    - [Relevance: 0.0500] The four waves of language models are statistical language models, neural language models, pre-trained language models, and LLMs.
    - [Relevance: 0.0400] The survey aims to prove a valuable and accessible resource for students, researchers and developers.
    - [Relevance: 0.0300] The recent success of LLMs is an accumulation of decades of research and development of language models.
    - [Relevance: 0.0300] Language models can be

In [4]:
from graphrag_toolkit.lexical_graph.retrieval.model import SearchResult

def get_query_params_for_results(response, include_sources=True, include_facts=True, limit=-1):

    statement_ids = []
    source_params = []
    fact_params = []
    
    nodes = response[:limit] if isinstance(response, list) else response.source_nodes[:limit]
    
    for n in nodes:
        
        search_result = SearchResult.model_validate(n.metadata)
        source_id = search_result.source.sourceId
        
        for topic in search_result.topics:
            
            for statement in topic.statements:
                
                statement_id = statement.statementId
                chunk_id = statement.chunkId
                
                statement_ids.append(statement_id)
                if include_sources:
                    source_params.append({'s': source_id, 'c': chunk_id, 'l': statement_id})
                if include_facts:
                    fact_params.append(statement_id)
                    
    
    query_parameters = { 
        'statement_ids': statement_ids,
        'source_params': source_params,
        'fact_params': fact_params
    }
    
    return query_parameters
    
query_parameters = get_query_params_for_results(response, limit=10)

In [9]:
# # Test graph-notebook import
# import graph_notebook
# print(f"Graph Notebook version: {graph_notebook.__version__}")
# print("✅ Graph Notebook imported successfully!")

In [10]:
# # Load graph-notebook magic commands
# %load_ext graph_notebook.magics
# print("✅ Graph Notebook magic commands loaded!")

In [11]:
display_var = '{"__Source__":"url","__Chunk__":"value","__Topic__":"value","__Statement__":"value","__Fact__":"value"}'


In [12]:
%%oc --query-parameters query_parameters -d $display_var -l 20

UNWIND $source_params AS source_params
MATCH p=(s:`__Source__`)<--(c:`__Chunk__`)<--(t:`__Topic__`)<--(l:`__Statement__`)
WHERE id(s) = source_params.s 
    AND id(c) = source_params.c 
    AND id(l) = source_params.l
RETURN p
UNION
MATCH p=(x:`__Source__`)<--(:`__Chunk__`)<--(:`__Topic__`)<--(l:`__Statement__`)<-[:`__SUPPORTS__`]-(:`__Fact__`)-[:`__NEXT__`*0..1]->(:`__Fact__`)-[:`__SUPPORTS__`]->(ll:`__Statement__`)-->(:`__Topic__`)-->(:`__Chunk__`)-->(y:`__Source__`)
WHERE id(l) IN $fact_params
    AND id(ll) IN $fact_params
    AND x <> y
RETURN p
UNION
MATCH p=(l:`__Statement__`)
WHERE id(l) IN $statement_ids
RETURN p

UsageError: Cell magic `%%oc` not found.
